# 多 Agent Supervisor：可验证路由与上下文交接

**面试问题：Supervisor 怎样把任务交给合适专家，并避免上下文丢失和循环转派？**

## 回答主线

1. Supervisor 的核心不是让另一个 LLM 随意聊天，而是基于意图、风险和能力做显式路由。
2. Handoff 应传递结构化信封：目标、事实、证据、约束、已尝试动作和期望下一步。
3. 接收方必须验证必需字段和证据权限，缺失时退回补充而不是猜测。
4. 每次转派都记录发送者、接收者、原因和状态，才能定位责任与延迟。
5. visited 集合和最大 hop 预算用于终止乒乓路由，复杂高风险任务应升级人工。
6. 评估要同时看路由准确率、上下文完整率、任务成功率和转派次数。

## 真实案例

六张企业客服工单覆盖账单、账号安全、物流、退款风控、技术 API 和模糊问题。三个专家 Agent 有明确能力和风险上限；我们先用单关键词路由，再实现 Supervisor 评分、handoff 信封校验和循环保护。数据是离线脱敏教学样本，只用于解释机制，不能宣称线上收益。

### 输入预览：六张工单与专家能力

In [1]:
tickets = [  # 构造六张跨领域和风险等级的工单。
    {"id": "T1", "text": "发票金额与合同不一致", "intent": "billing", "risk": 1, "evidence": ["invoice-88", "contract-12"]},  # 账单核对任务。
    {"id": "T2", "text": "账号疑似被盗需要冻结", "intent": "security", "risk": 3, "evidence": ["login-alert-5"]},  # 高风险安全任务。
    {"id": "T3", "text": "包裹三天没有更新", "intent": "shipping", "risk": 1, "evidence": ["tracking-S9"]},  # 物流查询任务。
    {"id": "T4", "text": "退款8999元需要复核", "intent": "billing", "risk": 3, "evidence": ["order-A7", "risk-R2"]},  # 高风险账单任务。
    {"id": "T5", "text": "API签名总是401", "intent": "technical", "risk": 2, "evidence": ["trace-401"]},  # 技术支持任务。
    {"id": "T6", "text": "这个问题谁能处理", "intent": "ambiguous", "risk": 2, "evidence": []},  # 故意构造模糊路由任务。
]  # 完成工单集合。
agents = {  # 定义专家能力和可处理风险上限。
    "billing_agent": {"skills": {"billing", "refund"}, "max_risk": 2},  # 普通账单专家不能独立处理高额退款。
    "security_agent": {"skills": {"security"}, "max_risk": 3},  # 安全专家可处理高风险账号事件。
    "ops_agent": {"skills": {"shipping", "technical"}, "max_risk": 2},  # 运维专家处理物流和 API。
}  # 完成能力注册表。
print("工单  intent      risk  文本                         证据")  # 输出路由输入表头。
for ticket in tickets:  # 逐工单展示真实字段。
    print(f"{ticket['id']}   {ticket['intent']:<11} {ticket['risk']:>2}   {ticket['text']:<27} {ticket['evidence']}")  # 展示领域、风险和证据。

工单  intent      risk  文本                         证据
T1   billing      1   发票金额与合同不一致                  ['invoice-88', 'contract-12']
T2   security     3   账号疑似被盗需要冻结                  ['login-alert-5']
T3   shipping     1   包裹三天没有更新                    ['tracking-S9']
T4   billing      3   退款8999元需要复核                 ['order-A7', 'risk-R2']
T5   technical    2   API签名总是401                  ['trace-401']
T6   ambiguous    2   这个问题谁能处理                    []


## Baseline 基线：首个关键词命中即路由

In [2]:
keyword_routes = {"发票": "billing_agent", "退款": "billing_agent", "账号": "security_agent", "包裹": "ops_agent", "API": "ops_agent"}  # 定义脆弱的单关键词表。
def keyword_route(text):  # 返回第一个命中关键词的专家。
    for keyword, agent in keyword_routes.items():  # 按字典顺序扫描关键词。
        if keyword in text:  # 当前工单包含关键词时立即结束。
            return agent  # 返回对应专家而不检查风险和证据。
    return "billing_agent"  # 模糊问题错误默认给账单专家。

baseline_routes = [keyword_route(ticket["text"]) for ticket in tickets]  # 对六张工单执行基线路由。
print("工单  基线路由          风险可承受")  # 输出基线路由表头。
for ticket, route in zip(tickets, baseline_routes):  # 逐工单检查专家风险上限。
    risk_ok = ticket["risk"] <= agents[route]["max_risk"]  # 判断被选专家能否承担风险。
    print(f"{ticket['id']}   {route:<18} {risk_ok}")  # 展示 T4 和模糊任务的问题。

工单  基线路由          风险可承受
T1   billing_agent      True
T2   security_agent     True
T3   ops_agent          True
T4   billing_agent      False
T5   ops_agent          True
T6   billing_agent      True


### 核心实现：能力/风险评分与结构化 Handoff 信封

In [3]:
def supervisor_route(ticket):  # 根据能力、风险和歧义做确定性路由。
    candidates = []  # 收集满足领域能力的专家。
    for name, profile in agents.items():  # 遍历专家注册表。
        if ticket["intent"] in profile["skills"] and ticket["risk"] <= profile["max_risk"]:  # 同时要求技能匹配和风险可承受。
            candidates.append(name)  # 加入合格候选。
    if len(candidates) == 1:  # 唯一专家可以直接接单。
        return candidates[0], "capability-and-risk-match"  # 返回可解释路由原因。
    if ticket["risk"] >= 3 or not candidates:  # 高风险无专家或语义模糊时升级人工。
        return "human_queue", "no-safe-unique-agent"  # 避免默认猜测。
    return sorted(candidates)[0], "deterministic-tie-break"  # 多个安全候选时稳定打破平局。

def make_handoff(ticket, receiver, reason):  # 构造可验证的专家交接信封。
    return {"task_id": ticket["id"], "sender": "supervisor", "receiver": receiver, "goal": ticket["text"], "facts": {"intent": ticket["intent"], "risk": ticket["risk"]}, "evidence": ticket["evidence"], "constraints": ["不得越权写操作", "高风险需人工审批"], "attempts": [], "next_action": "inspect-evidence", "reason": reason, "schema_version": 1}  # 返回完整上下文合同。

def validate_handoff(envelope):  # 检查接收方执行前需要的字段。
    required = {"task_id", "sender", "receiver", "goal", "facts", "evidence", "constraints", "next_action", "schema_version"}  # 定义必需字段集合。
    missing = sorted(required - set(envelope))  # 计算缺失字段。
    evidence_required = envelope.get("facts", {}).get("risk", 0) >= 2  # 中高风险任务必须携带证据。
    reasons = [f"missing:{field}" for field in missing]  # 把结构缺失转换为拒绝原因。
    if evidence_required and not envelope.get("evidence"):  # 检查高风险证据是否为空。
        reasons.append("missing-risk-evidence")  # 保存语义完整性错误。
    return len(reasons) == 0, reasons  # 返回接收或退回结论。

supervisor_rows = []  # 收集六张工单的路由和信封验证结果。
for ticket in tickets:  # 逐工单运行 Supervisor。
    receiver, reason = supervisor_route(ticket)  # 选择安全接收方。
    envelope = make_handoff(ticket, receiver, reason)  # 生成结构化交接信封。
    valid, reasons = validate_handoff(envelope)  # 由接收侧验证上下文。
    supervisor_rows.append({"id": ticket["id"], "receiver": receiver, "reason": reason, "valid": valid, "errors": reasons, "envelope": envelope})  # 保存路由账本。
print("T4 Handoff：", supervisor_rows[3]["envelope"])  # 展示高额退款被升级人工的完整上下文。

T4 Handoff： {'task_id': 'T4', 'sender': 'supervisor', 'receiver': 'human_queue', 'goal': '退款8999元需要复核', 'facts': {'intent': 'billing', 'risk': 3}, 'evidence': ['order-A7', 'risk-R2'], 'constraints': ['不得越权写操作', '高风险需人工审批'], 'attempts': [], 'next_action': 'inspect-evidence', 'reason': 'no-safe-unique-agent', 'schema_version': 1}


## 结果解读：路由安全与上下文完整率

In [4]:
expected_routes = ["billing_agent", "security_agent", "ops_agent", "human_queue", "ops_agent", "human_queue"]  # 定义领域和风险共同决定的期望接收方。
baseline_accuracy = sum(actual == expected for actual, expected in zip(baseline_routes, expected_routes)) / len(tickets)  # 计算关键词路由准确率。
supervisor_accuracy = sum(row["receiver"] == expected for row, expected in zip(supervisor_rows, expected_routes)) / len(tickets)  # 计算 Supervisor 路由准确率。
context_rate = sum(row["valid"] for row in supervisor_rows) / len(supervisor_rows)  # 计算信封结构完整率。
print("工单  基线路由          Supervisor         原因")  # 输出同口径路由对照表头。
for ticket, baseline, row in zip(tickets, baseline_routes, supervisor_rows):  # 逐工单展示路由变化。
    print(f"{ticket['id']}   {baseline:<18} {row['receiver']:<18} {row['reason']}")  # 展示高风险与模糊任务升级。
print(f"路由准确率 {baseline_accuracy:.1%} -> {supervisor_accuracy:.1%}，Handoff 完整率={context_rate:.1%}")  # 汇总教学指标。
print("解读：T4 虽然领域是 billing，但风险超过专家上限；T6 没有唯一能力匹配和证据，因此安全答案是升级而非随便转派。")  # 解释策略。

工单  基线路由          Supervisor         原因
T1   billing_agent      billing_agent      capability-and-risk-match
T2   security_agent     security_agent     capability-and-risk-match
T3   ops_agent          ops_agent          capability-and-risk-match
T4   billing_agent      human_queue        no-safe-unique-agent
T5   ops_agent          ops_agent          capability-and-risk-match
T6   billing_agent      human_queue        no-safe-unique-agent
路由准确率 66.7% -> 100.0%，Handoff 完整率=83.3%
解读：T4 虽然领域是 billing，但风险超过专家上限；T6 没有唯一能力匹配和证据，因此安全答案是升级而非随便转派。


## 失败案例：两个专家对模糊任务反复转派

In [5]:
def route_with_loop_guard(start_agent, proposals, max_hops):  # 用 visited 集合和 hop 预算执行转派。
    visited = []  # 保存已经处理过任务的 Agent。
    current = start_agent  # 从 Supervisor 的初始选择开始。
    for hop in range(max_hops):  # 限制最多转派次数。
        if current in visited:  # 再次进入同一 Agent 表示乒乓循环。
            return "human_queue", visited + [current], "cycle-detected"  # 立即升级人工并保留循环路径。
        visited.append(current)  # 标记当前 Agent 已访问。
        current = proposals.get(current, "done")  # 读取当前 Agent 建议的下一接收方。
        if current == "done":  # 专家已完成任务时正常终止。
            return visited[-1], visited, "completed"  # 返回完成者和路径。
    return "human_queue", visited, "hop-budget-exhausted"  # 超过预算时升级人工。

ping_pong = {"billing_agent": "ops_agent", "ops_agent": "billing_agent"}  # 构造两个专家互相认为对方负责的循环。
final_receiver, handoff_path, stop_reason = route_with_loop_guard("billing_agent", ping_pong, max_hops=4)  # 执行带循环保护的转派。
print(f"循环路径={handoff_path}，最终接收方={final_receiver}，停止原因={stop_reason}")  # 展示第三跳前检测到重复节点。
print("修正策略：handoff 信封携带 visited_agents 和 remaining_hops；重复节点或预算耗尽必须升级，不允许重写 task_id 逃逸。")  # 给出终止合同。

循环路径=['billing_agent', 'ops_agent', 'billing_agent']，最终接收方=human_queue，停止原因=cycle-detected
修正策略：handoff 信封携带 visited_agents 和 remaining_hops；重复节点或预算耗尽必须升级，不允许重写 task_id 逃逸。


### 生产边界与审计事件

In [6]:
handoff_event = {"task_id": "T4", "from": "supervisor", "to": "human_queue", "reason": "no-safe-unique-agent", "risk": 3, "evidence_count": 2, "schema_version": 1}  # 构造高风险路由审计事件。
print("Handoff 事件：", handoff_event)  # 展示可回放字段。
print("生产替换点：真实系统还需学习式意图分类、动态能力注册、证据 ACL、并发状态、SLA、人工队列和端到端任务成功评估。")  # 明确规则路由边界。

Handoff 事件： {'task_id': 'T4', 'from': 'supervisor', 'to': 'human_queue', 'reason': 'no-safe-unique-agent', 'risk': 3, 'evidence_count': 2, 'schema_version': 1}
生产替换点：真实系统还需学习式意图分类、动态能力注册、证据 ACL、并发状态、SLA、人工队列和端到端任务成功评估。


## 回归测试：最后只保护安全路由、上下文和循环终止

In [7]:
assert supervisor_accuracy == 1.0 and supervisor_accuracy > baseline_accuracy  # 验证能力/风险路由优于关键词基线。
assert all(row["valid"] for row in supervisor_rows if row["receiver"] != "human_queue") and supervisor_rows[5]["errors"] == ["missing-risk-evidence"]  # 验证专家交接完整且模糊高风险任务因无证据被明确标记。
assert supervisor_rows[3]["receiver"] == "human_queue" and supervisor_rows[3]["envelope"]["evidence"]  # 验证高额退款携证据升级人工。
assert final_receiver == "human_queue" and stop_reason == "cycle-detected"  # 验证乒乓转派被循环保护终止。
assert handoff_path == ["billing_agent", "ops_agent", "billing_agent"]  # 验证审计路径保留首次重复节点。
print("回归测试通过：路由准确率、风险升级、信封完整、证据传递和循环终止均成立。")  # 用少量断言总结 Handoff 合同。

回归测试通过：路由准确率、风险升级、信封完整、证据传递和循环终止均成立。
